In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Number of spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    print("Feature scaling complete")

    return X_scaled, X_test_scaled


# -----------------------------
# Hyperparameter search
# -----------------------------
def hyperparameter_search(X, y):

    print("\nStarting hyperparameter search...\n")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    alphas = [0.01, 0.1, 1, 5, 10]
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    best_score = np.inf
    best_params = None

    results = []

    for alpha in alphas:
        for l1 in l1_ratios:

            rmse_scores = []

            for train_idx, val_idx in kf.split(X):

                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                model = ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1,
                    max_iter=100000,
                    tol=1e-3,
                    random_state=42
                )

                model.fit(X_train, y_train)

                preds = model.predict(X_val)

                rmse = np.sqrt(mean_squared_error(y_val, preds))
                rmse_scores.append(rmse)

            mean_rmse = np.mean(rmse_scores)

            results.append((alpha, l1, mean_rmse))

            print(f"alpha={alpha:<5}  l1_ratio={l1:<4}  RMSE={mean_rmse:.4f}")

            if mean_rmse < best_score:
                best_score = mean_rmse
                best_params = (alpha, l1)

    print("\nBest parameters found:")
    print("alpha =", best_params[0])
    print("l1_ratio =", best_params[1])
    print("Best CV RMSE =", best_score)

    return best_params


# -----------------------------
# Train final model
# -----------------------------
def train_final_model(X, y, X_test, alpha, l1_ratio):

    print("\nTraining final model...")

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X, y)

    preds = model.predict(X_test)

    print("Sample predictions:", preds[:10])

    return preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp09_elasticnet_hyperparameter_search_20260323"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = scale_features(X, X_test)

    best_alpha, best_l1 = hyperparameter_search(X, y)

    preds = train_final_model(X, y, X_test, best_alpha, best_l1)

    save_submission(test, preds)


# Run experiment
main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Feature scaling complete

Starting hyperparameter search...

alpha=0.01   l1_ratio=0.1   RMSE=17.3631
alpha=0.01   l1_ratio=0.3   RMSE=17.1434
alpha=0.01   l1_ratio=0.5   RMSE=17.0042
alpha=0.01   l1_ratio=0.7   RMSE=16.6022
alpha=0.01   l1_ratio=0.9   RMSE=15.9025
alpha=0.1    l1_ratio=0.1   RMSE=20.0371
alpha=0.1    l1_ratio=0.3   RMSE=20.0451
alpha=0.1    l1_ratio=0.5   RMSE=19.9906
alpha=0.1    l1_ratio=0.7   RMSE=19.9091
alpha=0.1    l1_ratio=0.9   RMSE=19.7824
alpha=1      l1_ratio=0.1   RMSE=23.4204
alpha=1      l1_ratio=0.3   RMSE=23.6780
alpha=1      l1_ratio=0.5   RMSE=23.8857
alpha=1      l1_ratio=0.7   RMSE=24.1042
alpha=1      l1_ratio=0.9   RMSE=24.2399
alpha=5      l1_ratio=0.1   RMSE=28.0302
alpha=5      l1_ratio=0.3   RMSE=30.1839
alpha=5      l1_ratio=0.5   RMSE=31.5331
alpha=5      l1_ratio=0.7   RMSE=31.5146
alpha=5      l1_ratio=0.9   RMSE=31.5121
alpha=10     l1_rat